<a href="https://colab.research.google.com/github/Patro331/MscMak2025-IntroductionToPython/blob/main/Deep_Learning_for_Health_Data_Chest_X_ray_Inference_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Deep Learning for Health Data - Demo
# Chest X-ray inference with prediction scores
# ============================================

!pip -q install torchxrayvision scikit-image pillow matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchxrayvision as xrv

# -----------------------------
# 1) Load pretrained chest X-ray model
# -----------------------------
print("Loading pretrained model...")
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model.eval()
print("Model loaded successfully.")

# -----------------------------
# 2) Get image
# -----------------------------
local_image_path = "sample_cxr.png"

if not os.path.exists(local_image_path):
    print(f"File '{local_image_path}' not found.")
    print("Please upload a chest X-ray image.")

    from google.colab import files
    uploaded = files.upload()

    if len(uploaded) == 0:
        raise ValueError("No image was uploaded.")

    local_image_path = list(uploaded.keys())[0]

print(f"Using image: {local_image_path}")

# -----------------------------
# 3) Load image as grayscale
# -----------------------------
image = Image.open(local_image_path).convert("L")
img_array = np.array(image)

print("Original image shape:", img_array.shape)

# -----------------------------
# 4) Preprocess image
# -----------------------------
# Normalize
img = xrv.datasets.normalize(img_array, 255)

# Make sure image is H x W
if img.ndim > 2:
    img = img[:, :, 0]

# Add channel dimension -> shape becomes (1, H, W)
img = img[None, :, :]

print("Shape after adding channel dimension:", img.shape)

# Center crop and resize
transform = xrv.datasets.XRayCenterCrop()
resizer = xrv.datasets.XRayResizer(224)

img = transform(img)
img = resizer(img)

print("Shape after crop/resize:", img.shape)

# Convert to tensor and add batch dimension -> (1, 1, 224, 224)
img = torch.from_numpy(img).unsqueeze(0).float()

print("Processed tensor shape:", img.shape)

# -----------------------------
# 5) Run inference
# -----------------------------
with torch.no_grad():
    outputs = model(img)[0]

preds = outputs.detach().cpu().numpy()
pathologies = model.pathologies

# -----------------------------
# 6) Sort predictions
# -----------------------------
pairs = list(zip(pathologies, preds))
pairs = sorted(pairs, key=lambda x: float(x[1]), reverse=True)

# -----------------------------
# 7) Print top predictions
# -----------------------------
print("\nTop predicted findings:")
for name, score in pairs[:8]:
    print(f"{name:20s} {float(score):.3f}")

# -----------------------------
# 8) Show original image
# -----------------------------
plt.figure(figsize=(6, 6))
plt.imshow(image, cmap="gray")
plt.title("Input Chest X-ray")
plt.axis("off")
plt.show()

# -----------------------------
# 9) Plot top prediction scores
# -----------------------------
top_k = 8
top_names = [p[0] for p in pairs[:top_k]][::-1]
top_scores = [float(p[1]) for p in pairs[:top_k]][::-1]

plt.figure(figsize=(8, 5))
plt.barh(top_names, top_scores)
plt.xlabel("Model score")
plt.title("Top Predicted Findings")
plt.show()

# Student Assignment: Exploring a Pretrained Chest X-ray Model

## Objective
In this exercise, you will use a pretrained deep learning model to analyze chest X-ray images and interpret its outputs. The goal is not to train a model, but to understand how **inference**, **preprocessing**, and **prediction scores** work in medical AI.

---

## Instructions

### Task 1: Run the notebook successfully
Run all the cells in the notebook using at least **one chest X-ray image**.

Make sure your notebook displays:
- the original image,
- the processed tensor shape,
- the top predicted findings,
- the bar chart of prediction scores.

---

### Task 2: Try at least two different images
Run the notebook on **two different chest X-ray images**.

For each image, record:
- the file name,
- the original image shape,
- the top 5 predicted findings and their scores.

Then compare the results.

#### Questions
1. Did the top predictions change between the two images?
2. Which image produced higher scores?
3. Do the outputs appear reasonable based on the image?

---

### Task 3: Explain the preprocessing steps
Using this notebook, explain the purpose of each of the following steps:

- converting the image to grayscale,
- normalizing the image,
- adding the channel dimension,
- center cropping,
- resizing,
- converting the image to a tensor.

Write your explanation in simple language.

---

### Task 4: Explain what the model is doing
In your own words, answer the following:

1. What is the difference between **training** and **inference**?
2. Is this notebook training a model or using a trained model?
3. What is the purpose of `model.eval()`?
4. Why do we use `torch.no_grad()`?
5. Why does the model return several findings instead of just one?

---

### Task 5: Interpret the prediction scores
Look at the output below the heading **Top predicted findings**.

For your first image, answer:
1. Which finding had the highest score?
2. What was the score?
3. Do you think this score means the model is certain? Explain.
4. Why should these outputs not be treated as a final diagnosis?

---

### Task 6: Modify the notebook
Make **at least two** of the following changes to the notebook:

- change the number of displayed predictions from 8 to 5,
- change the plot title,
- print only predictions with score greater than 0.50,
- sort and display the results in a table,
- save the bar chart as an image file,
- display the uploaded image size before preprocessing.

Clearly indicate in markdown what you changed.

---

### Task 7: Critical reflection
Write a short reflection of about **150–250 words** addressing the following:

- What did you learn from this notebook?
- What are the strengths of using pretrained models?
- What are the limitations of this demo in a real healthcare setting?
- What additional steps would be needed before such a model could be used in practice?

---

## Expected Submission
Submit:
1. Your completed notebook  
2. A short PDF or Word document answering the written questions  
3. Output screenshots for **at least two images**

---

## Marking Guide

| Component | Marks |
|---|---:|
| Successfully running the notebook | 15 |
| Using and comparing two images | 20 |
| Explanation of preprocessing | 15 |
| Explanation of inference and model behavior | 15 |
| Interpretation of prediction scores | 10 |
| Notebook modifications | 10 |
| Critical reflection | 15 |
| **Total** | **100** |

---

## Optional Bonus
Try one of the following for bonus marks:
- test the notebook with a very low-quality image and compare results,
- add confidence values directly onto the plot,
- convert the predictions into a pandas DataFrame,
- discuss whether the uploaded image is truly a chest X-ray and why image quality matters.

---

## Note
In this notebook, we used a pretrained chest X-ray model to perform inference on an uploaded image. The aim of this assignment is to help you understand how medical image AI pipelines work, including input preparation, model inference, output interpretation, and critical reflection on limitations.